# 🌱 Plant Disease Detection Project

This notebook classifies plant diseases using the PlantVillage dataset and MobileNetV2 architecture.

In [ ]:
# 1. SETUP & IMPORTS
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import splitfolders
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# 2. DATA PREPARATION
# Please upload 'archive.zip' to the same folder before running this cell.
zip_path = "archive.zip"
extract_path = "PlantVillage_Raw"
split_path = "PlantVillage_Split"

if os.path.exists(zip_path):
    if not os.path.exists(extract_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Dataset extracted.")
    
    input_folder = os.path.join(extract_path, "PlantVillage")
    if not os.path.exists(split_path):
        splitfolders.ratio(input_folder, output=split_path, seed=42, ratio=(.8, .1, .1), group_prefix=None, move=False)
        print("Data split complete.")
else:
    print("WARNING: archive.zip not found. Please upload the dataset.")

In [ ]:
# 3. DATA LOADING
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = "PlantVillage_Split"

if os.path.exists(DATA_DIR):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'train'),
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'val'),
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'test'),
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
    )
    
    class_names = train_ds.class_names
    print(f"Classes: {len(class_names)}")
    
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
else:
    print("Data directory not found. Steps skipped.")

In [ ]:
# 4. MODEL BUILDING (MobileNetV2)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
])

base_model = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
# Assuming 38 classes, dynamic based on data
outputs = layers.Dense(38, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 5. TRAINING
# Only runs if data is present
if os.path.exists(DATA_DIR):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
    )
    
    # Plotting
    plt.plot(history.history['accuracy'], label='accuracy')
    plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.show()